# Prepare Dataset

In [1]:
import json, string
import numpy as np

In [2]:
with open("data/alfred/train.json", "r") as file:
    train_dataset = json.load(file)

with open("data/alfred/valid_unseen.json", "r") as file:
    test_dataset = json.load(file)

with open("data/stopwords.json", "r") as file:
    stopwords = set(json.load(file))

def text_to_words(text):
    return text.strip(string.punctuation + string.whitespace).lower().split()

# Method 1: Bag of Words (BoW)

For each label, consider all the task descriptions as its document content.
In BoW model, we simply count the occurences of each word, without considering the order.

In [3]:
from collections import defaultdict, Counter

# Construct the BoW for each label
BoW = defaultdict(Counter)
for item in train_dataset:
    BoW[item["label"]].update(text_to_words(item["task"]))

# Print the labels
for label in BoW.keys():
    print(label)

look_at_obj_in_light
pick_and_place_simple
pick_and_place_with_movable_recep
pick_clean_then_place_in_recep
pick_cool_then_place_in_recep
pick_heat_then_place_in_recep
pick_two_obj_and_place


## 1.1 BoW Probability

First we consider predicting by the probability of words in each labels. That is:

$$
P(\text{text}|\text{label}) 
= \prod_{\text{word}\in\text{text}} p(\text{label}|\text{word})
= \prod_{\text{word}\in\text{text}} \frac{\text{tf}_{\text{word},\text{label}}}{\text{tf}_{\text{word}}}
$$

Obviously for calculation simplicity we can simply compare:

$$
\begin{aligned}
&\log P(\text{text}|\text{label}) \\
=& \sum_{\text{word}\in\text{text}} \left( \log{\text{tf}_{\text{word},\text{label}}} -\log{\text{tf}_{\text{word}}} \right) \\
=& \sum_{\text{word}\in\text{text}} \log{\text{tf}_{\text{word},\text{label}}}\ 
\underbrace{-\sum_{\text{word}\in\text{text}} \log{\text{tf}_{\text{word}}}}_\text{common part}
\end{aligned}
$$

In [11]:
def bow_prob(text, remove_stopwords=False):
    words = text_to_words(text)
    label_log_probs = {}
    for label, counter in BoW.items():
        log_prob = 0.0
        for word in words:
            if remove_stopwords and word in stopwords:
                continue
            if counter[word] > 0:
                log_prob += np.log(counter[word])
        label_log_probs[label] = log_prob
    return max(label_log_probs, key=label_log_probs.get)

In [ ]:
correct = 0
for item in test_dataset:
    label = item["label"]
    predict = bow_prob(item["task"], remove_stopwords=False)
    if label == predict:
        correct += 1
print(f"Accuracy: {correct / len(test_dataset)}")

Accuracy: 0.9013398294762485
